## Import

In [26]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import os

In [ ]:
# load data
train_data = pd.read_csv("data/samsum-train.csv")
val_data = pd.read_csv("data/samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
print(train_data.shape)
print(val_data.shape)

(14732, 3)
(818, 3)


In [7]:
# random sampling
train_data =  train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data =  val_data.sample(n=500, random_state=42).reset_index(drop=True)

## Data Preprocessing

In [8]:
# Data Preprocessing

import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags
    test = text.strip().lower()
    return text

In [9]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [10]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet: Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## Tokenize

#### t5-small
- its is a lightwight version of T5 Model
- because of fatser and light for us

In [11]:
# download tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#### tokenizer
- have **padding, max-length, truncate**

#### tokenizer return
```
{
    "input_id": [...],
    "attention_mask": [...]
}
```

In [12]:
# raw data => tokenize inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # summary_tokens_id => add into inputs dictionary as labels

    return inputs

In [13]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

# we convert to list because it is campatible for HuggingFace

In [14]:
train_dataset[0] 

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

#### inputs_dataset : for a single index
```py
{
    'input_ids': [...],  # dialogue => token ids
    'attention_mask': [...],
    'labels': [...] # summary => tokens ids
}
```

1 -> EOS : end of seq
0 -> padding

#### attention_mask
- show that which val is valid or which one is padding val
- based on that we decide which vals are not going to consider

#### type 
type(train_dataset) -> list  
type(val_dataset)  -> list

In [15]:
len(train_dataset[0]["input_ids"])
len(train_dataset[0]["labels"])

150

## Work with model - fine tune

In [16]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device :", device)
model.to(device)

device : cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

### Defining some training arguments

- we use **TraningArguments**: its is a transformer model

- some default params already set in: [class transormers.TrainingArguments](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments)

- 


In [18]:
# training argumants

training_args = TrainingArguments(
    output_dir =  "./results",

    num_train_epochs = 6,
    weight_decay = 0.1,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch", # evaluate after evry single epoch
    save_strategy="epoch", # we want to save our model after evry epoch

    warmup_steps=500
    # 0 => lr default
)

### Define a [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) for train our model

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.021768,0.388812
2,0.408142,0.366634
3,0.384193,0.359568
4,0.371819,0.356127
5,0.364727,0.354676
6,0.360407,0.354487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9851758778889974, metrics={'train_runtime': 1443.2239, 'train_samples_per_second': 16.629, 'train_steps_per_second': 2.079, 'total_flos': 3248203235328000.0, 'train_loss': 0.9851758778889974, 'epoch': 6.0})

In [ ]:
# full process of Hugging face model use
# model load => fine-tune => save the model

**Note**: if you want to save the model on your own device:
```py
{
    model.save_pretrained("./saved_summary_model")
    tokenizer.save_pretrained("./saved_summary_model")
}
```

or if you use **Collab GPU** than follow it
1. **mount drive** - because I use Collab GPU on **VS-code** 
2. **save model** : collab => google drive
    - we are working inside Collab (remote enviornment)
    - that filesystem (/content, /tmp, etc.) exits on google servers
    - So all our saved files stored inside **cloud VM**, not our laptop/pc.

In [27]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# define/make google drive model_path
model_path = "/content/drive/MyDrive/Projects/text_summarizer/saved_summary_model"

os.makedirs(model_path, exist_ok=True)

In [30]:
# save model
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Projects/text_summarizer/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/Projects/text_summarizer/saved_summary_model/tokenizer.json')

In [32]:
# load model
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## test the code logic for summarization

In [43]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt" # it returns - Pytorch tensors as inputs
        # by default Hugging face model is Pytorch model
    ).to(device)

    # generate the summary => token ids (generate token ids, not actual summary text)
    model.to(device) # ensure that our data and model on same device
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4, # transformer generate 4 seq of out(summary) -> and choose one best
        early_stopping=True # after get EOS -> stop: and choose one best
    )
    
    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # 
    return summary

In [41]:
test_sum_data = pd.read_csv("data/samsum-test.csv")

In [35]:
test_sum_data["dialogue"][0]

"Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye"

In [44]:
test_dialogue = "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye"

summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)


Summary:  Amanda has Betty's number. She can't find it. Larry called her last time she was at the park together.
